# A3.8 · Environment separation

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

Builds on **[A3.7 · Runtime containment levers](https://spbreed.github.io/cyber-commons/lessons/A3.7.html)**.

| | |
|---|---|
| Open-source tooling | SPIRE, OPA |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Environment separation for ordinary services is a network problem: dev cannot
reach prod because a VPC boundary says so.

Agents break that model in a specific way: **an agent carries its context across
boundaries.** The same agent, the same prompt, the same conversation history can
be pointed at dev on Monday and prod on Tuesday. Worse, an agent debugging a
production incident legitimately needs to read production while running from a
development context.

So the separation has to bind to the **identity and its policy**, which travels
with the agent, rather than to the network location, which does not.

Three properties a working separation has:

1. **Distinct identities per environment.** `agent@dev` and `agent@prod` are
   different principals with different ceilings, not one agent with a flag.
2. **The workspace is part of the identity's policy**, so a dev agent's path
   guard cannot resolve into prod paths regardless of where the process runs.
3. **Cross-environment access is a JIT grant with a reason**, never a second
   standing credential. That is A2.8 doing its job here.

## 2 · Demo — two environments, two identities, one agent image

In [ ]:
import fnmatch, time
from dataclasses import dataclass, field

def normalise(p):
    parts = []
    for seg in p.split("/"):
        if seg in ("", "."): continue
        if seg == "..":
            if parts: parts.pop()
            continue
        parts.append(seg)
    return "/" + "/".join(parts)

@dataclass
class EnvIdentity:
    """Identity and policy travel together — that is the whole design."""
    name: str
    workspace: str
    allow_hosts: set
    allow_tools: set
    deny_tools: set = field(default_factory=set)
    deny_globs: tuple = ("*/.ssh/*", "*/.aws/*", "*.pem", "*/.env")

    def read(self, path):
        real = normalise(path)
        for g in self.deny_globs:
            if fnmatch.fnmatch(real, g): return False, f"deny rule {g}"
        ws = normalise(self.workspace)
        if real == ws or real.startswith(ws + "/"):
            return True, f"inside {self.name} workspace"
        return False, f"outside {self.name} workspace (resolves to {real})"

    def tool(self, t):
        if t in self.deny_tools: return False, "denied in this environment"
        if t in self.allow_tools: return True, "permitted"
        return False, "not on this environment's tool allowlist"

DEV = EnvIdentity("dev", "/work/dev",
                  allow_hosts={"api.github.com", "dev-api.internal"},
                  allow_tools={"read_file", "write_file", "run_shell", "http_get"})
PROD = EnvIdentity("prod", "/work/prod",
                   allow_hosts={"api.github.com"},
                   allow_tools={"read_file", "http_get"},
                   deny_tools={"run_shell", "write_file"})

for env in (DEV, PROD):
    print(f"--- {env.name} ---")
    for path in (f"/work/{env.name}/app.py", "/work/prod/config/secrets.yaml",
                 "/work/dev/../prod/db.conf"):
        ok, why = env.read(path)
        print(f"   read  {path:36s} {'ALLOW' if ok else 'DENY '} {why}")
    for t in ("read_file", "run_shell"):
        ok, why = env.tool(t)
        print(f"   tool  {t:36s} {'ALLOW' if ok else 'DENY '} {why}")
    print()

## 3 · Where it breaks — the same process, pointed at prod

The failure this design prevents: an agent running in the dev cluster that is handed a prod endpoint. On a network-only separation, if the route exists the agent is in. Here the policy travels with the identity, so location is irrelevant.

In [ ]:
def run_agent(identity, requests, where):
    print(f"agent process running in {where}, holding identity '{identity.name}'")
    for kind, arg in requests:
        ok, why = (identity.read(arg) if kind == "read" else identity.tool(kind))
        print(f"   {kind:10s} {arg[:34]:36s} {'ALLOW' if ok else 'DENY '} {why}")

reqs = [("read", "/work/prod/config/secrets.yaml"),
        ("read", "/work/dev/../prod/db.conf"),
        ("run_shell", "")]

run_agent(DEV, reqs, where="the DEV cluster")
print()
run_agent(DEV, reqs, where="the PROD cluster (misconfigured deployment)")
print("\nIdentical results. The agent moved; its authority did not.")
print("A network-only separation would have permitted all three in the second case.")

## 4 · The control — cross-environment access as a JIT grant

The legitimate case still has to work: an engineer needs the agent to read production logs during an incident. The wrong answer is a second standing credential. The right one is A2.8 — a bounded, justified, expiring grant.

In [ ]:
class GrantExpired(Exception): pass

@dataclass
class CrossEnvGrant:
    actor: str
    from_env: str
    to_env: str
    paths: tuple                    # narrow, not the whole environment
    reason: str
    ttl: float
    granted: float = field(default_factory=time.time)
    @property
    def active(self): return time.time() - self.granted < self.ttl
    def permits(self, path):
        if not self.active:
            raise GrantExpired(f"grant expired after {self.ttl}s ({self.reason!r})")
        real = normalise(path)
        return any(fnmatch.fnmatch(real, p) for p in self.paths)

def read_with_grant(identity, path, grant=None):
    ok, why = identity.read(path)
    if ok: return True, why
    if grant and grant.actor == identity.name + "-agent":
        try:
            if grant.permits(path):
                return True, (f"cross-env JIT grant: {grant.from_env}→{grant.to_env} "
                              f"reason={grant.reason!r}")
        except GrantExpired as e:
            return False, str(e)
    return False, why

g = CrossEnvGrant("dev-agent", "dev", "prod",
                  paths=("/work/prod/logs/*",),          # logs only, not secrets
                  reason="INC-2291 payments latency", ttl=0.6)

for path in ("/work/prod/logs/payments.log", "/work/prod/config/secrets.yaml"):
    ok, why = read_with_grant(DEV, path, g)
    print(f"{'ALLOW' if ok else 'DENY ':5s} {path:36s} {why}")

time.sleep(0.7)
print("\nafter the grant expires:")
for path in ("/work/prod/logs/payments.log",):
    ok, why = read_with_grant(DEV, path, g)
    print(f"{'ALLOW' if ok else 'DENY ':5s} {path:36s} {why}")

In [ ]:
# Verify: no path traversal can cross environments, grant or no grant.
import random
random.seed(5)
SEG = ["work", "dev", "prod", "..", ".", "config", "logs", "secrets.yaml", "app.py"]
leaks = []
for _ in range(20000):
    p = "/" + "/".join(random.choice(SEG) for _ in range(random.randint(1, 7)))
    ok, _ = DEV.read(p)
    if ok and not normalise(p).startswith("/work/dev"):
        leaks.append(p)
print(f"20000 random paths against the dev identity — cross-env leaks: {len(leaks)}")
assert not leaks
print("Separation holds under traversal, and it holds wherever the process runs.")

## What you just proved

The dev identity permits its own workspace and refuses both the prod secrets path and the traversal into prod; the prod identity denies `run_shell` and `write_file` outright. Running the dev identity inside the prod cluster produces identical decisions. The JIT grant permits the prod log path but not the secrets path, and stops permitting anything once it expires. The 20,000-path property test finds zero cross-environment leaks.

## Your turn

Check whether your dev and prod agents are two identities or one identity with an environment variable. If it is the latter, the separation is a config flag, and a config flag is not a boundary.

---

**Next → [A3.9 · The unmanaged agent problem](https://spbreed.github.io/cyber-commons/lessons/A3.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*